In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from bs4 import BeautifulSoup

# Setup WebDriver (Automatically finds the correct driver)
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

In [5]:
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")  # Opens browser in full-screen
options.add_argument("--disable-blink-features=AutomationControlled")  # Avoid detection
options.add_experimental_option("detach", True)  # Keeps browser open
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome() 

# Open the Google login page
driver.get("https://accounts.google.com/signin")

# Wait for the page to load
time.sleep(5)

# Find and fill in the email field
email_input = driver.find_element(By.ID, "identifierId")
email_input.send_keys("joel.tchapnda@aims.ac.rw")  # Replace with your email
email_input.send_keys(Keys.RETURN)
time.sleep(5)

In [8]:
stathead_url = "https://stathead.com/fbref/player-match-finder.cgi?request=1&timeframe=seasons&height_type=height_meters&order_by=xg&year_min=2024-2025&weight_type=kgs&force_min_year=1&year_max=2024-2025&match_status=1&is_qualifier=0&comp_type=b5&comp_gender=m&match=player_game&offset=0"
driver.get(stathead_url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source

In [9]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

# Find the first table
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        data = [cell.text.strip() for cell in cells]
        print(data)
#else:
    #print("No table found on the page!")



['', '', '', '', 'Performance', 'Expected', 'Standard', '', 'Rk', 'Player', 'xG', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'xG', 'npxG', 'xAG', 'xG+xAG', 'xA', 'npxG+xAG', 'G-xG', 'np:G-xG', 'A-xAG', 'npxG/Sh', 'Sh', 'G/Sh', 'G/SoT', 'SoT', 'SoT%', 'Dist', 'FK', 'Pos.', 'Match Report']


In [19]:
exp_cols= data[8:]

In [12]:
body= table.find_all("tbody")
rows= body[0].find_all("tr")
row_data= []
for row in rows:
    cells= row.find_all(["th", "td"])
    data= [cell.text.strip() for cell in cells]
    if data[0] not in ['', 'Rk']:
        row_data.append(data) 
    print(data)

['1', 'Serhou Guirassy', '2.8', '2025-02-22', '28-347', 'gn GUI', 'Dortmund', '', 'Union Berlin', 'de Bundesliga', 'W 6-0', 'Y', '84', '4', '0', '4', '4', '0', '0', '0', '2.8', '2.8', '0.0', '2.8', '0.0', '2.8', '+1.2', '+1.2', '0.0', '0.34', '8', '0.50', '1.00', '4', '50.0', '8.5', '0', 'FW', 'Match Report']
['2', 'Harry Kane', '2.6', '2024-11-22', '31-117', 'eng ENG', 'Bayern Munich', '', 'Augsburg', 'de Bundesliga', 'W 3-0', 'Y', '90', '3', '0', '3', '1', '2', '2', '0', '2.6', '1.0', '0.0', '2.6', '0.2', '1.0', '+0.4', '0.0', '0.0', '0.25', '4', '0.25', '0.50', '2', '50.0', '17.0', '0', 'FW', 'Match Report']
['3', 'Justin Kluivert', '2.4', '2024-11-30', '25-209', 'nl NED', 'Bournemouth', '@', 'Wolves', 'eng Premier League', 'W 4-2', 'Y', '76', '3', '0', '3', '0', '3', '3', '0', '2.4', '0.0', '0.5', '2.9', '0.1', '0.6', '+0.6', '0.0', '-0.5', '0.05', '1', '0.00', '0.00', '1', '100.0', '57.9', '0', 'MF,FW', 'Match Report']
['4', 'Cole Palmer', '2.4', '2024-09-28', '22-145', 'eng ENG',

In [13]:
len(row_data)

200

In [16]:
expected_stats_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&timeframe=seasons&height_type=height_meters&order_by=xg&year_min=2024-2025&weight_type=kgs&force_min_year=1&year_max=2024-2025&match_status=1&is_qualifier=0&comp_type=b5&comp_gender=m&match=player_game&offset="
row_exp_data= []
for page in range(0, 35200, 200):
    url= expected_stats_url + str(page)
    driver.get(url)

    time.sleep(5)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    time.sleep(3)
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_exp_data.append(data) 

In [17]:
len(row_exp_data)

35200

In [20]:
df_xG= pd.DataFrame(row_exp_data, columns= exp_cols)
df_xG

ValueError: 31 columns passed, passed data had 39 columns

In [ ]:
df_xG.to_csv("../Data/Expected Stats.csv", index= False)

In [21]:
passing_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&order_by=passes_completed&force_min_year=1&timeframe=seasons&weight_type=kgs&height_type=height_meters&match_status=1&comp_type=b5&year_max=2024-2025&is_qualifier=0&comp_gender=m&match=player_game&year_min=2024-2025&offset="
row_data= []
url= passing_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_data = [cell.text.strip() for cell in cells]
        print(data)

['35200', 'Santiago Bueno', '0.0', '2024-12-04', '26-025', 'uy URU', 'Wolves', '@', 'Everton', 'eng Premier League', 'L 0-4', 'Y', '76', '0', '0', '0', '0', '0', '0', '0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '', '0', '', '', '0', '', '', '0', 'DF', 'Match Report']


In [24]:
cols_pas= cols_data[13:]
cols_pas

['Rk',
 'Player',
 'Cmp',
 'Date',
 'Age',
 'Nation',
 'Team',
 '',
 'Opp',
 'Comp',
 'Result',
 'Start',
 'Min',
 'Gls',
 'Ast',
 'G+A',
 'G-PK',
 'PK',
 'PKatt',
 'PKm',
 'Cmp',
 'Att',
 'Cmp%',
 'KP',
 '1/3',
 'PPA',
 'CrsPA',
 'PrgP',
 'TotDist',
 'PrgDist',
 'Cmp',
 'Att',
 'Cmp%',
 'Cmp',
 'Att',
 'Cmp%',
 'Cmp',
 'Att',
 'Cmp%',
 'Pos.',
 'Match Report']

In [25]:
row_pas_data= []
for page in range(0, 35200, 200):
    url= passing_url + str(page)
    driver.get(url)

    time.sleep(8)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_pas_data.append(data) 

In [26]:
cols_pas[24]= "Last 3rd Pa"

In [27]:
for i in range(30, 33):
    cols_pas[i]= cols_pas[i]+ " 5-15"

In [28]:
for i in range(33, 36):
    cols_pas[i]= cols_pas[i]+ " 15-30"

for i in range(36, 39):
    cols_pas[i]= cols_pas[i]+ " 30+"

In [29]:
cols

['Rk',
 'Player',
 'xG',
 'Date',
 'Age',
 'Nation',
 'Team',
 '',
 'Opp',
 'Comp',
 'Result',
 'Start',
 'Min',
 'Gls',
 'Ast',
 'G+A',
 'G-PK',
 'PK',
 'PKatt',
 'PKm',
 'xG',
 'npxG',
 'xAG',
 'xG+xAG',
 'xA',
 'npxG+xAG',
 'G-xG',
 'np:G-xG',
 'A-xAG',
 'npxG/Sh',
 'Sh',
 'G/Sh',
 'G/SoT',
 'SoT',
 'SoT%',
 'Dist',
 'FK',
 'Pos.',
 'Match Report']

In [30]:
df_Pa= pd.DataFrame(row_pas_data, columns= cols_pas)
df_Pa

,Rk,Player,Cmp,Date,Age,Nation,Team,,Opp,Comp,...,Att 5-15,Cmp% 5-15,Cmp 15-30,Att 15-30,Cmp% 15-30,Cmp 30+,Att 30+,Cmp% 30+,Pos.,Match Report
0,1,Waldemar Anton,171,2024-10-18,28-090,de GER,Dortmund,,St. Pauli,de Bundesliga,...,57,98.2,105,109,96.3,10,17,58.8,DF,Match Report
1,2,Nico Schlotterbeck,161,2024-10-18,24-322,de GER,Dortmund,,St. Pauli,de Bundesliga,...,41,100.0,106,106,100.0,13,15,86.7,DF,Match Report
2,3,Joshua Kimmich,160,2025-02-07,29-365,de GER,Bayern Munich,,Werder Bremen,de Bundesliga,...,80,96.3,66,68,97.1,14,19,73.7,MF,Match Report
3,4,Vitinha,159,2024-11-30,24-291,pt POR,Paris S-G,,Nantes,fr Ligue 1,...,69,92.8,75,78,96.2,17,18,94.4,MF,Match Report
4,5,Emre Can,149,2025-01-14,31-002,de GER,Dortmund,@,Holstein Kiel,de Bundesliga,...,73,91.8,76,80,95.0,5,7,71.4,"DF,MF",Match Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35195,35196,Giovanni Fabbian,2,2025-02-22,22-039,it ITA,Bologna,@,Parma,it Serie A,...,3,66.7,0,0,,0,0,,MF,Match Report
35196,35197,Richarlison,2,2024-10-19,27-162,br BRA,Tottenham,,West Ham,eng Premier League,...,0,,2,3,66.7,0,0,,FW,Match Report
35197,35198,Yannik Engelhardt,2,2025-02-07,24-000,de GER,Como,,Juventus,it Serie A,...,2,100.0,0,0,,0,0,,"MF,DF",Match Report
35198,35199,Jeff Ekhator,2,2024-09-15,17-309,it ITA,Genoa,,Roma,it Serie A,...,1,100.0,0,0,,0,0,,"FW,MF",Match Report


In [31]:
df_Pa.to_csv("../Data/Passing.csv", index= False)

In [32]:
shot_crating_action_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&comp_gender=m&weight_type=kgs&comp_type=b5&order_by=sca&year_max=2024-2025&match=player_game&match_status=1&is_qualifier=0&year_min=2024-2025&timeframe=seasons&force_min_year=1&height_type=height_meters&offset="
url= shot_crating_action_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_data = [cell.text.strip() for cell in cells]
        print(cols_data)

['', '', '', '', 'Performance', '', 'SCA Types', '', 'Rk', 'Player', 'SCA', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'SCA', 'PassLive', 'PassDead', 'TO', 'Sh', 'Fld', 'Def', 'Pos.', 'Match Report']


In [33]:
row_sca_data= []
for page in range(0, 35200, 200):
    url= shot_crating_action_url + str(page)
    driver.get(url)

    time.sleep(8)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_sca_data.append(data) 

In [34]:
cols_sca= cols_data[8:]
cols_sca

['Rk',
 'Player',
 'SCA',
 'Date',
 'Age',
 'Nation',
 'Team',
 '',
 'Opp',
 'Comp',
 'Result',
 'Start',
 'Min',
 'Gls',
 'Ast',
 'G+A',
 'G-PK',
 'PK',
 'PKatt',
 'PKm',
 'SCA',
 'PassLive',
 'PassDead',
 'TO',
 'Sh',
 'Fld',
 'Def',
 'Pos.',
 'Match Report']

In [35]:
df_SCA= pd.DataFrame(row_sca_data, columns= cols_sca)
df_SCA

,Rk,Player,SCA,Date,Age,Nation,Team,,Opp,Comp,...,PKm,SCA,PassLive,PassDead,TO,Sh,Fld,Def,Pos.,Match Report
0,1,Kevin Stöger,21,2024-09-21,31-025,at AUT,Gladbach,@,Eint Frankfurt,de Bundesliga,...,0,21,6,9,2,0,4,0,"MF,FW",Match Report
1,2,Bukayo Saka,16,2024-09-28,23-023,eng ENG,Arsenal,,Leicester City,eng Premier League,...,0,16,10,3,1,2,0,0,"FW,MF",Match Report
2,3,Omar Marmoush,13,2024-09-29,25-235,eg EGY,Eint Frankfurt,@,Holstein Kiel,de Bundesliga,...,0,13,6,3,2,2,0,0,FW,Match Report
3,4,Ryan Christie,13,2024-12-16,29-298,sct SCO,Bournemouth,,West Ham,eng Premier League,...,0,13,9,2,0,1,0,1,MF,Match Report
4,5,Lamine Yamal,13,2024-08-27,17-045,es ESP,Barcelona,@,Rayo Vallecano,es La Liga,...,0,13,8,0,3,1,1,0,FW,Match Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35195,35196,Marco Brescianini,0,2024-11-23,24-308,it ITA,Atalanta,@,Parma,it Serie A,...,0,0,0,0,0,0,0,0,MF,Match Report
35196,35197,Leandro Paredes,0,2025-02-24,30-240,ar ARG,Roma,,Monza,it Serie A,...,0,0,0,0,0,0,0,0,MF,Match Report
35197,35198,Leandro Paredes,0,2024-11-10,30-134,ar ARG,Roma,,Bologna,it Serie A,...,0,0,0,0,0,0,0,0,MF,Match Report
35198,35199,Ismaël Koné,0,2024-10-20,22-136,ca CAN,Marseille,@,Montpellier,fr Ligue 1,...,0,0,0,0,0,0,0,0,"MF,FW",Match Report


In [36]:
df_SCA.to_csv("../Data/Shot-Creating Actions.csv", index= False)

In [37]:
goal_crating_action_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&order_by=gca&is_qualifier=0&match_status=1&year_min=2024-2025&year_max=2024-2025&timeframe=seasons&height_type=height_meters&match=player_game&force_min_year=1&weight_type=kgs&comp_gender=m&comp_type=b5&offset="
url= goal_crating_action_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_data = [cell.text.strip() for cell in cells]
        print(cols_data)

['', '', '', '', 'Performance', '', 'GCA Types', '', 'Rk', 'Player', 'GCA', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'GCA', 'PassLive', 'PassDead', 'TO', 'Sh', 'Fld', 'Def', 'Pos.', 'Match Report']


In [38]:
row_gca_data= []
for page in range(0, 35200, 200):
    url= goal_crating_action_url + str(page)
    driver.get(url)

    time.sleep(8)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_gca_data.append(data) 

In [39]:
cols_gca= cols_data[8:]

In [40]:
df_GCA= pd.DataFrame(row_gca_data, columns= cols_gca)
df_GCA

,Rk,Player,GCA,Date,Age,Nation,Team,,Opp,Comp,...,PKm,GCA,PassLive,PassDead,TO,Sh,Fld,Def,Pos.,Match Report
0,1,Bukayo Saka,6,2024-11-30,23-086,eng ENG,Arsenal,@,West Ham,eng Premier League,...,0,6,2,2,1,0,1,0,"FW,MF",Match Report
1,2,Pascal Groß,5,2025-02-22,33-252,de GER,Dortmund,,Union Berlin,de Bundesliga,...,0,5,4,0,0,0,0,1,"MF,DF",Match Report
2,3,Dominik Szoboszlai,5,2024-12-22,24-058,hu HUN,Liverpool,@,Tottenham,eng Premier League,...,0,5,3,0,0,2,0,0,MF,Match Report
3,4,Evann Guessand,5,2024-09-20,23-081,ci CIV,Nice,,Saint-Étienne,fr Ligue 1,...,0,5,3,0,1,1,0,0,"FW,MF",Match Report
4,5,Cole Palmer,5,2024-08-25,22-111,eng ENG,Chelsea,@,Wolves,eng Premier League,...,0,5,4,0,1,0,0,0,MF,Match Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35195,35196,José María Giménez,0,2025-02-08,30-019,uy URU,Atlético Madrid,@,Real Madrid,es La Liga,...,0,0,0,0,0,0,0,0,DF,Match Report
35196,35197,José María Giménez,0,2024-12-15,29-330,uy URU,Atlético Madrid,,Getafe,es La Liga,...,0,0,0,0,0,0,0,0,DF,Match Report
35197,35198,José María Giménez,0,2024-10-20,29-274,uy URU,Atlético Madrid,,Leganés,es La Liga,...,0,0,0,0,0,0,0,0,DF,Match Report
35198,35199,José María Giménez,0,2024-08-25,29-218,uy URU,Atlético Madrid,,Girona,es La Liga,...,0,0,0,0,0,0,0,0,DF,Match Report


In [41]:
df_GCA.to_csv("../Data/Goal-Creating Actions.csv", index= False)

In [42]:
carries_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&year_max=2024-2025&match=player_game&weight_type=kgs&comp_type=b5&order_by=carries&is_qualifier=0&height_type=height_meters&timeframe=seasons&comp_gender=m&match_status=1&force_min_year=1&year_min=2024-2025&offset="
url= carries_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_data_caries = [cell.text.strip() for cell in cells]
        print(cols_data_caries)

['', '', '', '', 'Performance', 'Carries', '', 'Carries', '', '', '', 'Rk', 'Player', 'Carries', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'Carries', 'TotDist', 'PrgDist', 'PrgC', '1/3', 'CPA', 'Mis', 'Dis', 'Rec', 'PrgR', 'Pos.', 'Match Report']


In [43]:
row_data_carries= []
for page in range(0, 35200, 200):
    url= carries_url + str(page)
    driver.get(url)

    time.sleep(8)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_carries.append(data) 

In [44]:
receiving_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&year_min=2024-2025&comp_gender=m&weight_type=kgs&comp_type=b5&is_qualifier=0&force_min_year=1&timeframe=seasons&match_status=1&year_max=2024-2025&match=player_game&order_by=passes_received&height_type=height_meters&offset="
url= receiving_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_data_receiving = [cell.text.strip() for cell in cells]
        print(cols_data_receiving)

['', '', '', '', 'Performance', '', '', 'Carries', '', 'Carries', '', 'Rk', 'Player', 'Rec', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'Rec', 'PrgR', 'Carries', 'TotDist', 'PrgDist', 'PrgC', '1/3', 'CPA', 'Mis', 'Dis', 'Pos.', 'Match Report']


In [45]:
row_data_receiving= []
for page in range(0, 35200, 200):
    url= receiving_url + str(page)
    driver.get(url)

    time.sleep(8)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_receiving.append(data) 

In [46]:
touches_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&year_min=2024-2025&weight_type=kgs&match=player_game&match_status=1&height_type=height_meters&timeframe=seasons&force_min_year=1&is_qualifier=0&year_max=2024-2025&comp_gender=m&comp_type=b5&order_by=touches&offset="
url= touches_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_data_touches = [cell.text.strip() for cell in cells]
        print(cols_data_touches)

['', '', '', '', 'Performance', 'Touches', '', 'Rk', 'Player', 'Touches', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'Touches', 'Def Pen', 'Def 3rd', 'Mid 3rd', 'Att 3rd', 'Att Pen', 'Live', 'Pos.', 'Match Report']


In [47]:
row_data_touches= []
for page in range(0, 35200, 200):
    url= touches_url + str(page)
    driver.get(url)

    time.sleep(8)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_touches.append(data) 

ReadTimeoutError: HTTPConnectionPool(host='localhost', port=58576): Read timed out. (read timeout=120)

In [ ]:
take_ons_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&weight_type=kgs&comp_type=b5&year_max=2024-2025&match=player_game&height_type=height_meters&timeframe=seasons&match_status=1&year_min=2024-2025&order_by=take_ons&is_qualifier=0&force_min_year=1&comp_gender=m&offset="
url= take_ons_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_data_take_ons = [cell.text.strip() for cell in cells]
        print(cols_data_take_ons)

['', '', '', '', 'Performance', 'Take-Ons', '', 'Rk', 'Player', 'Att', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'Att', 'Succ', 'Succ%', 'Tkld', 'Tkld%', 'Pos.', 'Match Report']


In [ ]:
row_data_take_ons= []
for page in range(0, 35200, 200):
    url= take_ons_url + str(page)
    driver.get(url)

    time.sleep(8)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_take_ons.append(data) 

In [ ]:
defensive_actions_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&order_by=tackles&force_min_year=1&comp_gender=m&is_qualifier=0&weight_type=kgs&match_status=1&year_max=2024-2025&height_type=height_meters&timeframe=seasons&comp_type=b5&match=player_game&year_min=2024-2025&offset="
url= defensive_actions_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_data_defensive_actions = [cell.text.strip() for cell in cells]
        print(cols_data_defensive_actions)

['', '', '', '', 'Performance', 'Tackles', 'Challenges', 'Blocks', '', 'Rk', 'Player', 'Tkl', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'Tkl', 'TklW', 'Def 3rd', 'Mid 3rd', 'Att 3rd', 'Tkl', 'Att', 'Tkl%', 'Lost', 'Blocks', 'Sh', 'Pass', 'Int', 'Tkl+Int', 'Clr', 'Err', 'Pos.', 'Match Report']


In [ ]:
row_data_defensive_actions= []
for page in range(0, 35200, 200):
    url= defensive_actions_url + str(page)
    driver.get(url)

    time.sleep(8)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_defensive_actions.append(data) 

In [ ]:
aeral_duels_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&is_qualifier=0&match=player_game&timeframe=seasons&comp_gender=m&weight_type=kgs&comp_type=b5&year_max=2024-2025&order_by=aerials_won&match_status=1&force_min_year=1&year_min=2024-2025&height_type=height_meters&offset="
url= aeral_duels_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_aeral_duels = [cell.text.strip() for cell in cells]
        print(cols_aeral_duels)

['', '', '', '', 'Performance', 'Aerial Duels', '', 'Rk', 'Player', 'Won', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'Won', 'Lost', 'Won%', 'Pos.', 'Match Report']


In [ ]:
row_data_aeral_duels= []
for page in range(0, 35200, 200):
    url= aeral_duels_url + str(page)
    driver.get(url)

    time.sleep(8)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_aeral_duels.append(data) 

In [ ]:
pass_types_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&year_max=2024-2025&is_qualifier=0&year_min=2024-2025&match=player_game&weight_type=kgs&comp_type=b5&order_by=passes_live&comp_gender=m&match_status=1&height_type=height_meters&timeframe=seasons&force_min_year=1&offset="
url= pass_types_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_pass_types = [cell.text.strip() for cell in cells]
        print(cols_pass_types)

['', '', '', '', 'Performance', 'Pass Types', 'Corner Kicks', 'Outcomes', '', 'Rk', 'Player', 'Live', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'Live', 'Dead', 'FK', 'TB', 'Sw', 'Crs', 'TI', 'CK', 'In', 'Out', 'Str', 'Off', 'Blocks', 'Pos.', 'Match Report']


In [ ]:
row_data_pass_types= []
for page in range(0, 35200, 200):
    url= pass_types_url + str(page)
    driver.get(url)

    time.sleep(8)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_pass_types.append(data) 

In [ ]:
cards_fouls_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&year_min=2024-2025&match=player_game&force_min_year=1&timeframe=seasons&comp_type=b5&comp_gender=m&weight_type=kgs&match_status=1&order_by=cards_yellow&height_type=height_meters&is_qualifier=0&year_max=2024-2025&offset="
url= cards_fouls_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_cards_fouls = [cell.text.strip() for cell in cells]
        print(cols_cards_fouls)

['', '', '', '', 'Performance', '', 'Rk', 'Player', 'CrdY', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'CrdY', 'CrdR', '2CrdY', 'Fls', 'Fld', 'Pos.', 'Match Report']


In [ ]:
row_data_cards_fouls= []
for page in range(0, 35200, 200):
    url= cards_fouls_url + str(page)
    driver.get(url)

    time.sleep(6)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_cards_fouls.append(data) 

In [ ]:
misc_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&comp_gender=m&year_min=2024-2025&force_min_year=1&weight_type=kgs&timeframe=seasons&comp_type=b5&year_max=2024-2025&match=player_game&is_qualifier=0&order_by=offsides&match_status=1&height_type=height_meters&offset="
url= misc_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_misc = [cell.text.strip() for cell in cells]
        print(cols_misc)

['', '', '', '', 'Performance', '', 'Rk', 'Player', 'Off', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'Off', 'PKwon', 'PKcon', 'OG', 'Recov', 'Pos.', 'Match Report']


In [ ]:
row_data_misc= []
for page in range(0, 35200, 200):
    url= misc_url + str(page)
    driver.get(url)

    time.sleep(6)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_misc.append(data) 

In [ ]:
basic_gk_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&timeframe=seasons&year_min=2024-2025&order_by=gk_goals_against&comp_type=b5&comp_gender=m&match=player_game&year_max=2024-2025&match_status=1&weight_type=kgs&force_min_year=1&height_type=height_meters&is_qualifier=0&offset="
url= basic_gk_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_basic_gk = [cell.text.strip() for cell in cells]
        print(cols_basic_gk)

['', '', '', '', 'Performance', '', 'Rk', 'Player', 'GA', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'GA', 'SoTA', 'Saves', 'Save%', 'Pos.', 'Match Report']


In [ ]:
row_data_basic_gk= []
for page in range(0, 35200, 200):
    url= basic_gk_url + str(page)
    driver.get(url)

    time.sleep(6)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_basic_gk.append(data) 

In [ ]:
pks_gk_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&force_min_year=1&match=player_game&comp_gender=m&is_qualifier=0&match_status=1&timeframe=seasons&year_min=2024-2025&order_by=gk_pens_att&height_type=height_meters&comp_type=b5&year_max=2024-2025&weight_type=kgs&offset="
url= pks_gk_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_pks_gk = [cell.text.strip() for cell in cells]
        print(cols_pks_gk)

['', '', '', '', 'Performance', 'Penalty Kicks', '', 'Rk', 'Player', 'PKatt', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'PKatt', 'PKA', 'PKsv', 'PKm', 'Save%', 'Pos.', 'Match Report']


In [ ]:
row_data_pks_gk= []
for page in range(0, 35200, 200):
    url= pks_gk_url + str(page)
    driver.get(url)

    time.sleep(6)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_pks_gk.append(data) 

In [ ]:
expected_stats_gk_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&match=player_game&order_by=gk_psxg&comp_gender=m&timeframe=seasons&year_max=2024-2025&comp_type=b5&weight_type=kgs&match_status=1&year_min=2024-2025&force_min_year=1&is_qualifier=0&height_type=height_meters&offset="
url= expected_stats_gk_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_expected_stats_gk = [cell.text.strip() for cell in cells]
        print(cols_expected_stats_gk)

['', '', '', '', 'Performance', 'Expected', '', '', 'Rk', 'Player', 'PSxG', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'PSxG', 'PSxG/SoT', 'PSxG+/-', 'GA', 'Pos.', 'Match Report']


In [ ]:
row_data_expected_stats_gk= []
for page in range(0, 35200, 200):
    url= expected_stats_gk_url + str(page)
    driver.get(url)

    time.sleep(6)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_expected_stats_gk.append(data) 

In [ ]:
passes_gk_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&match_status=1&force_min_year=1&comp_gender=m&year_min=2024-2025&height_type=height_meters&order_by=gk_passes&timeframe=seasons&weight_type=kgs&match=player_game&comp_type=b5&year_max=2024-2025&is_qualifier=0&offset="
url= passes_gk_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_passes_gk = [cell.text.strip() for cell in cells]
        print(cols_passes_gk)

['', '', '', '', 'Performance', 'Passes', '', 'Rk', 'Player', 'Att (GK)', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'Att (GK)', 'Thr', 'Launch%', 'AvgLen', 'Pos.', 'Match Report']


In [ ]:
row_data_passes_gk= []
for page in range(0, 35200, 200):
    url= passes_gk_url + str(page)
    driver.get(url)

    time.sleep(6)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_passes_gk.append(data) 

In [ ]:
goal_kicks_gk_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&comp_gender=m&order_by=gk_goal_kicks&match_status=1&year_max=2024-2025&timeframe=seasons&force_min_year=1&is_qualifier=0&comp_type=b5&weight_type=kgs&year_min=2024-2025&match=player_game&height_type=height_meters&offset="
url= goal_kicks_gk_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_goal_kicks_gk = [cell.text.strip() for cell in cells]
        print(cols_goal_kicks_gk)

['', '', '', '', 'Performance', 'Goal Kicks', '', 'Rk', 'Player', 'Att', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'Att', 'Launch%', 'AvgLen', 'Pos.', 'Match Report']


In [ ]:
row_data_goal_kicks_gk= []
for page in range(0, 35200, 200):
    url= goal_kicks_gk_url + str(page)
    driver.get(url)

    time.sleep(6)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_goal_kicks_gk.append(data) 

In [ ]:
crosses_gk_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&comp_gender=m&height_type=height_meters&weight_type=kgs&year_max=2024-2025&is_qualifier=0&order_by=gk_crosses&year_min=2024-2025&force_min_year=1&comp_type=b5&match_status=1&match=player_game&timeframe=seasons&offset="
url= crosses_gk_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_crosses_gk = [cell.text.strip() for cell in cells]
        print(cols_crosses_gk)

['', '', '', '', 'Performance', 'Crosses', '', 'Rk', 'Player', 'Opp', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', 'Opp', 'Stp', 'Stp%', 'Pos.', 'Match Report']


In [ ]:
row_data_crosses_gk= []
for page in range(0, 35200, 200):
    url= crosses_gk_url + str(page)
    driver.get(url)

    time.sleep(6)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_crosses_gk.append(data) 

In [ ]:
sweeper_gk_url= "https://stathead.com/fbref/player-match-finder.cgi?request=1&height_type=height_meters&timeframe=seasons&comp_gender=m&force_min_year=1&year_max=2024-2025&order_by=gk_def_actions_outside_pen_area&comp_type=b5&weight_type=kgs&is_qualifier=0&match=player_game&year_min=2024-2025&match_status=1&offset="
url= sweeper_gk_url + "0"
driver.get(url)

time.sleep(5)  # Wait for the page to load

# Get page source after login
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")

table
if table:
    head = table.find_all("thead")
    for row in head:
        cells = row.find_all("th")
        cols_sweeper_gk = [cell.text.strip() for cell in cells]
        print(cols_sweeper_gk)

['', '', '', '', 'Performance', 'Sweeper', '', 'Rk', 'Player', '#OPA', 'Date', 'Age', 'Nation', 'Team', '', 'Opp', 'Comp', 'Result', 'Start', 'Min', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'PKm', '#OPA', 'AvgDist', 'Pos.', 'Match Report']


In [ ]:
row_data_sweeper_gk= []
for page in range(0, 35200, 200):
    url= sweeper_gk_url + str(page)
    driver.get(url)

    time.sleep(6)  # Wait for the page to load

# Get page source after login
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    body= table.find_all("tbody")
    rows= body[0].find_all("tr")
    for row in rows:
        cells= row.find_all(["th", "td"])
        data= [cell.text.strip() for cell in cells]
        if data[0] not in ['', 'Rk']:
            row_data_sweeper_gk.append(data) 

In [ ]:
df

,Name,Age,City
0,Alice,25,New York
1,Bob,30,Los Angeles
2,Charlie,35,Chicago
3,David,40,Houston
